In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00


In [2]:
import os
import torch

# 1. Tự động tìm đường dẫn chứa file .pt
DATASET_PATH = ""
for root, dirs, files in os.walk("/kaggle/input"):
    if any(f.endswith('.pt') for f in files):
        DATASET_PATH = root
        break

if not DATASET_PATH:
    raise FileNotFoundError("Không tìm thấy thư mục chứa file .pt! Hãy kiểm tra lại mục Input bên phải.")

print(f"Đã tìm thấy dữ liệu tại: {DATASET_PATH}")

# ===== LOAD FILE LIST =====
batch_files = sorted(
    [f for f in os.listdir(DATASET_PATH) if f.endswith(".pt")],
    key=lambda x: int(x.replace("batch_", "").replace(".pt", ""))
)

print(f"Tổng cộng tìm thấy: {len(batch_files)} batches.")

# ===== LOAD 1 BATCH =====
first_batch_path = os.path.join(DATASET_PATH, batch_files[0])
print(f"Đang nạp file: {first_batch_path} ...")

batch_0_data = torch.load(first_batch_path, weights_only=False)

print(f"Số lượng đồ thị trong batch 0: {len(batch_0_data)}")

print("\n--- THÔNG TIN ĐỒ THỊ ĐẦU TIÊN ---")
graph = batch_0_data[0]

print(graph)
print("Summary:", getattr(graph, "summary", "No summary"))

Đã tìm thấy dữ liệu tại: /kaggle/input/datasets/xuantudo/codesearchnet-java-gnn-batches/graph_batches
Tổng cộng tìm thấy: 49 batches.
Đang nạp file: /kaggle/input/datasets/xuantudo/codesearchnet-java-gnn-batches/graph_batches/batch_0.pt ...
Số lượng đồ thị trong batch 0: 1000

--- THÔNG TIN ĐỒ THỊ ĐẦU TIÊN ---
Data(x=[237, 2], edge_index=[2, 906], edge_type=[906], summary='DynamicInstantiatable has some pre-nit strings it uses as for spec
that do not match nit-compiler.', idx=0)
Summary: DynamicInstantiatable has some pre-nit strings it uses as for spec
that do not match nit-compiler.


In [3]:
all_graphs = []

for file in batch_files:
    path = os.path.join(DATASET_PATH, file)
    batch = torch.load(path, weights_only=False)
    all_graphs.extend(batch)

print("Total graphs:", len(all_graphs))

Total graphs: 48817


### **Chuẩn bị cho training**: Đưa list all_graphs vào một DataLoader chuyên dụng của PyTorch Geometric 

In [4]:
from torch_geometric.loader import DataLoader
import random

# Shuffle (xáo trộn) dữ liệu ngẫu nhiên trước khi chia tập Train/Test
random.seed(42)
random.shuffle(all_graphs)

# Chia dữ liệu: 80% Train, 10% Valid, 10% Test
num_total = len(all_graphs)
train_size = int(0.8 * num_total)
valid_size = int(0.1 * num_total)

train_dataset = all_graphs[:train_size]
valid_dataset = all_graphs[train_size:train_size+valid_size]
test_dataset = all_graphs[train_size+valid_size:]

print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}, Test: {len(test_dataset)}")

# Tạo DataLoader (mini-batch) để đưa vào GPU
# batch_size ở đây là số lượng đồ thị đưa vào GPU cùng 1 lúc (thường là 32, 64, 128)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Đã tạo DataLoader thành công! Dữ liệu sẵn sàng để đưa vào Mạng nơ-ron.")

Train: 39053, Valid: 4881, Test: 4883
Đã tạo DataLoader thành công! Dữ liệu sẵn sàng để đưa vào Mạng nơ-ron.


In [5]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=3877464 sha256=12d2eb609c7ea18d1b66f4d5155abf8fbb34011d532a9a8dbc5b3f8fc20a62da
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
Successfully built torch-scatter


**define GGNN Encoder architecture**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool

from transformers import AutoTokenizer

import json
import os
import random
import math


In [7]:
class GGNNLayer(nn.Module):
    def __init__(self, hidden_dim, num_edge_types):
        super().__init__()

        self.edge_embed = nn.Embedding(num_edge_types, hidden_dim)
        self.msg_proj = nn.Linear(hidden_dim, hidden_dim)

        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        # NEW
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, edge_index, edge_type):
        src, dst = edge_index

        h_src = x[src]
        e_emb = self.edge_embed(edge_type)

        msg = torch.relu(self.msg_proj(h_src) + e_emb)

        agg = scatter_add(msg, dst, dim=0, dim_size=x.size(0))

        x_new = self.gru(agg, x)

        # residual + norm
        x = self.norm(x + x_new)

        return x

In [8]:
class GraphEncoder(nn.Module):
    def __init__(self, num_node_types, hidden_dim, num_edge_types):
        super().__init__()

        self.embedding = nn.Embedding(num_node_types, hidden_dim)

        self.layers = nn.ModuleList([
            GGNNLayer(hidden_dim, num_edge_types)
            for _ in range(4)
        ])

    def forward(self, data):
        x = data.x[:, 0].long()
        x = self.embedding(x)

        for layer in self.layers:
            x = layer(x, data.edge_index, data.edge_type)

        return x

In [9]:
from torch_geometric.utils import to_dense_batch

def encode_graph_batch(node_emb, batch):
    """
    node_emb: [total_nodes, H]
    batch: [total_nodes]

    return:
        dense_x: [B, max_nodes, H]
        mask: [B, max_nodes]
    """
    dense_x, mask = to_dense_batch(node_emb, batch)
    return dense_x, mask

In [10]:
# ==========================================
# 1. POSITIONAL ENCODING CHUẨN SIN/COS
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [Batch, Seq_len, Embedding_dim]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

**Transformer Decoder**

In [11]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        
        # SỬ DỤNG POSITIONAL ENCODING SIN/COS THAY VÌ PARAMETER RANDOM
        self.pos_encoder = PositionalEncoding(d_model=hidden_dim, dropout=0.1)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=4,
            batch_first=True,
            dropout=0.1 # Thêm dropout để tránh overfitting
        )

        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=4)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt_ids, memory, memory_mask):
        B, T = tgt_ids.size()

        # ĐÃ FIX LỖI BUG: Sử dụng scaled_emb đúng cách
        scaled_emb = self.embedding(tgt_ids) * math.sqrt(self.embedding.embedding_dim)
        tgt_emb = self.pos_encoder(scaled_emb)

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T).to(tgt_ids.device)
        memory_key_padding_mask = ~memory_mask  

        out = self.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.fc(out)

**Graph2Seq (GGNN + Transformer)**

In [12]:
class Graph2SeqModel(nn.Module):
    def __init__(self, num_node_types, vocab_size, hidden_dim=128):
        super().__init__()

        self.encoder = GraphEncoder(num_node_types, hidden_dim, num_edge_types=3)

        # # THÊM POSITIONAL CHO GRAPH
        # self.graph_pos_encoder = PositionalEncoding(hidden_dim, dropout=0.1)

        self.decoder = TransformerDecoder(vocab_size, hidden_dim)

    def forward(self, data, input_ids):
        node_emb = self.encoder(data)

        memory, mask = encode_graph_batch(node_emb, data.batch)

        # # FIX QUAN TRỌNG: positional cho graph
        # memory = self.graph_pos_encoder(memory)

        logits = self.decoder(input_ids, memory, mask)

        return logits

In [13]:
# ==========================================
# 3. KHỞI TẠO MÔ HÌNH VÀ OPTIMIZER (KHÔNG BỊ TRÙNG LẶP)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

# Khởi tạo model (Đảm bảo class GraphEncoder và Graph2SeqModel đã được định nghĩa ở trên)
model = Graph2SeqModel(num_node_types=100, vocab_size=tokenizer.vocab_size, hidden_dim=128).to(device)

# KHAI BÁO DUY NHẤT 1 LẦN VỚI LR TỐI ƯU
optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id, label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-5
)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/tmp/ipykernel_23/2690031852.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


**Training Loop**

In [14]:
from torch_scatter import scatter_add
from torch.amp import autocast
# ==========================================
# 4. TRAINING LOOP CHUẨN KHÔNG CẦN CHỈNH
# ==========================================
EPOCHS = 30
ACCUM_STEPS = 2

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch+1} =====")

    model.train()
    total_loss = 0
    optimizer.zero_grad() # Dịch chuyển lên trước vòng lặp batch

    for i, batch in enumerate(train_loader):
        batch = batch.to(device)

        encoded = tokenizer(
            batch.summary, padding=True, truncation=True,
            max_length=64, return_tensors="pt"
        )
        target_ids = encoded["input_ids"].to(device)

        input_ids = target_ids[:, :-1]
        labels = target_ids[:, 1:]

        with autocast(device_type='cuda'):
            outputs = model(batch, input_ids)
            loss = criterion(
                outputs.reshape(-1, tokenizer.vocab_size),
                labels.reshape(-1)
            )
            # ĐÃ FIX BUG: Bắt buộc chia loss cho ACCUM_STEPS
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Nhân lại để thống kê in ra màn hình đúng giá trị thật
        total_loss += (loss.item() * ACCUM_STEPS)

    print("Train Loss:", total_loss / len(train_loader))

    # ===== VALID =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(device)
            encoded = tokenizer(
                batch.summary, padding=True, truncation=True,
                max_length=64, return_tensors="pt"
            )
            target_ids = encoded["input_ids"].to(device)

            input_ids = target_ids[:, :-1]
            labels = target_ids[:, 1:]

            # Không dùng autocast ở bước eval để tính số thực chính xác nhất
            outputs = model(batch, input_ids)
            loss = criterion(
                outputs.reshape(-1, tokenizer.vocab_size),
                labels.reshape(-1)
            )
            val_loss += loss.item()

    avg_val_loss = val_loss / len(valid_loader)
    print("Val Loss:", avg_val_loss)
    scheduler.step(avg_val_loss)


===== EPOCH 1 =====
Train Loss: 6.015130524553304
Val Loss: 5.096379753810908

===== EPOCH 2 =====
Train Loss: 4.931968836975723
Val Loss: 4.683308023253297

===== EPOCH 3 =====
Train Loss: 4.597059742336289
Val Loss: 4.471525413538116

===== EPOCH 4 =====
Train Loss: 4.381745671757316
Val Loss: 4.321628854165669

===== EPOCH 5 =====
Train Loss: 4.217838969804731
Val Loss: 4.217094436969632

===== EPOCH 6 =====
Train Loss: 4.091364885333324
Val Loss: 4.146374470268199

===== EPOCH 7 =====
Train Loss: 3.982479596118474
Val Loss: 4.095750278896755

===== EPOCH 8 =====
Train Loss: 3.889467602963334
Val Loss: 4.047457319459105

===== EPOCH 9 =====
Train Loss: 3.8086869357356665
Val Loss: 4.019098660525153

===== EPOCH 10 =====
Train Loss: 3.738676269556244
Val Loss: 3.9932274117189297

===== EPOCH 11 =====
Train Loss: 3.675710699189207
Val Loss: 3.971455946467281

===== EPOCH 12 =====
Train Loss: 3.618723740644088
Val Loss: 3.957674395804312

===== EPOCH 13 =====
Train Loss: 3.56814085822

**Beam Search**

In [15]:
def beam_search(model, graph, tokenizer, beam_size=3, max_len=32):
    model.eval()
    graph = graph.to(device)

    with torch.no_grad():
        node_emb = model.encoder(graph)
        memory, mask = encode_graph_batch(node_emb, graph.batch)

        sequences = [[[], 0.0]]

        for _ in range(max_len):
            all_candidates = []

            for seq, score in sequences:
                input_ids = torch.tensor([seq], device=device)

                outputs = model.decoder(input_ids, memory, mask)
                probs = torch.log_softmax(outputs[:, -1, :], dim=-1)

                topk = torch.topk(probs, beam_size)

                for i in range(beam_size):
                    token = topk.indices[0][i].item()
                    candidate = [seq + [token], score + topk.values[0][i].item()]
                    all_candidates.append(candidate)

            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_size]

        return tokenizer.decode(sequences[0][0])

In [16]:
!pip install rouge-score

In [17]:
import torch
import nltk
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

# ==========================================
# 1. HÀM INFERENCE NÂNG CẤP (XỬ LÝ SONG SONG)
# ==========================================
def generate_summary(model, graph_batch, tokenizer, device, max_len=32):
    model.eval()
    graph_batch = graph_batch.to(device)

    with torch.no_grad():
        # Encode toàn bộ batch đồ thị
        node_emb = model.encoder(graph_batch)
        memory, mask = encode_graph_batch(node_emb, graph_batch.batch)

        # TỰ ĐỘNG LẤY BATCH SIZE (Số lượng đồ thị thực tế trong lô)
        B = memory.size(0) 

        # Khởi tạo token [CLS] cho tất cả các đồ thị trong lô
        input_ids = torch.tensor([[tokenizer.cls_token_id]] * B).to(device)

        for _ in range(max_len):
            logits = model.decoder(input_ids, memory, mask)
            next_token_logits = logits[:, -1, :]
            
            # Tham lam (Greedy)
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # Nối từ mới vào chuỗi
            input_ids = torch.cat([input_ids, next_token_id], dim=1)

    # Giải mã toàn bộ batch thành danh sách các câu string
    return tokenizer.batch_decode(input_ids, skip_special_tokens=True)

# ==========================================
# 2. VÒNG LẶP CHẤM ĐIỂM TOÀN DIỆN
# ==========================================
def evaluate_model(model, test_loader, tokenizer, device):
    model.eval()
    chencherry = SmoothingFunction()
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    bleu_scores, meteor_scores, rouge_l_scores = [], [], []
    
    print("🚀 Bắt đầu đánh giá trên tập Test (Batch Inference)...")
    
    with torch.no_grad():
        for idx, batch in enumerate(test_loader):
            batch = batch.to(device)
            actual_summaries = batch.summary  # Đây là 1 list các câu chuẩn
            
            # Sinh ra 1 list các câu dự đoán tương ứng
            predicted_summaries = generate_summary(model, batch, tokenizer, device)
            
            # Chấm điểm từng cặp trong Lô
            for i in range(len(actual_summaries)):
                actual_summary = actual_summaries[i]
                predicted_summary = predicted_summaries[i]
                
                # Tách từ để đưa vào các hàm NLTK
                ref_tokens = nltk.word_tokenize(actual_summary.lower())
                pred_tokens = nltk.word_tokenize(predicted_summary.lower())
                
                if len(pred_tokens) == 0 or len(ref_tokens) == 0:
                    continue
                    
                bleu = sentence_bleu([ref_tokens], pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=chencherry.method1)
                meteor = meteor_score([ref_tokens], pred_tokens)
                rouge = scorer.score(actual_summary, predicted_summary)
                
                bleu_scores.append(bleu)
                meteor_scores.append(meteor)
                rouge_l_scores.append(rouge['rougeL'].fmeasure)
                
                # In 5 mẫu đầu tiên của lô đầu tiên làm Case Study cho báo cáo Word
                if idx == 0 and i < 5: 
                    print(f"\n[Case Study {i+1}]")
                    print(f"👉 Thực tế : {actual_summary}")
                    print(f"🤖 Dự đoán : {predicted_summary}")
                    print(f"   BLEU-4: {bleu:.4f} | METEOR: {meteor:.4f} | ROUGE-L: {rouge['rougeL'].fmeasure:.4f}")

    print("\n" + "="*40)
    print("🏆 KẾT QUẢ ĐÁNH GIÁ TỔNG THỂ (Trung bình) 🏆")
    print(f"BLEU-4  : {np.mean(bleu_scores) * 100:.2f}")
    print(f"METEOR  : {np.mean(meteor_scores) * 100:.2f}")
    print(f"ROUGE-L : {np.mean(rouge_l_scores) * 100:.2f}")
    print("="*40)

# Chạy đánh giá
evaluate_model(model, test_loader, tokenizer, device)

🚀 Bắt đầu đánh giá trên tập Test (Batch Inference)...

[Case Study 1]
👉 Thực tế : Returns a counter that is the intersection of c1 and c2.
🤖 Dự đoán : Returns a counter mapping from the counter.............
   BLEU-4: 0.0896 | METEOR: 0.2721 | ROUGE-L: 0.4444

[Case Study 2]
👉 Thực tế : Use this API to count the filtered set of linkset_channel_binding resources.
🤖 Dự đoán : Use this API to count the filtered set of nstrafficdomain_binding resources._binding resources._binding resources._binding resources.
   BLEU-4: 0.5969 | METEOR: 0.8916 | ROUGE-L: 0.7097

[Case Study 3]
👉 Thực tế : Deletes the server context.
🤖 Dự đoán : /*
(non-Javadoc)

@see org.#removeListener(byte[],[],[],[],[],
   BLEU-4: 0.0000 | METEOR: 0.0000 | ROUGE-L: 0.0000

[Case Study 4]
👉 Thực tế : Calculate Mean value.
🤖 Dự đoán : Returns the index of the first occurrence of the specified character, similar to the given character........
   BLEU-4: 0.0000 | METEOR: 0.0000 | ROUGE-L: 0.0000

[Case Study 5]
👉 Thực tế : 